## Merge function: shots + ASR + captions

In [3]:
import os
import json

def build_shot_evidence(video_name,
                        segmentation_dir,
                        asr_dir,
                        captions_dir,
                        output_dir):

    # ---- Paths ----
    video_folder = os.path.join(segmentation_dir, video_name)
    shots_path = os.path.join(video_folder, "shots.json")
    asr_path = os.path.join(asr_dir, f"{video_name}.json")
    captions_path = os.path.join(captions_dir, f"{video_name}.json")

    # ---- Load data ----
    with open(shots_path, "r") as f:
        shots = json.load(f)

    asr_segments = json.load(open(asr_path)) if os.path.exists(asr_path) else []
    captions = json.load(open(captions_path)) if os.path.exists(captions_path) else {}

    # ---- Align ASR to shots ----
    shot_asr = {int(k): "" for k in shots.keys()}

    for seg in asr_segments:
        for sid, shot in shots.items():
            if seg["start"] < shot["end"] and seg["end"] > shot["start"]:
                shot_asr[int(sid)] += " " + seg["text"]

    # ---- Build shot evidence ----
    evidence = {}

    for sid, shot in shots.items():
        sid_int = int(sid)

        evidence[sid] = {
            "shot_id": sid_int,
            "start": shot["start"],
            "end": shot["end"],
            "asr_text": shot_asr[sid_int].strip(),
            "visual_caption": captions.get(str(sid_int), ""),
            "keyframe": f"keyframes/shot_{sid_int:03d}.jpg"
        }

    # ---- Save ----
    os.makedirs(output_dir, exist_ok=True)
    out_path = os.path.join(output_dir, f"{video_name}.json")

    with open(out_path, "w", encoding="utf-8") as f:
        json.dump(evidence, f, indent=2)

    return out_path


## Batch merge over all videos

In [5]:
SEGMENTATION_DIR = "../data/segmentation"
ASR_DIR = "../data/asr"
CAPTIONS_DIR = "../data/captions"
EVIDENCE_DIR = "../data/shot_evidence"

for video_name in os.listdir(SEGMENTATION_DIR):
    video_path = os.path.join(SEGMENTATION_DIR, video_name)
    if not os.path.isdir(video_path):
        continue

    print(f"[MERGE] {video_name}")
    build_shot_evidence(
        video_name,
        SEGMENTATION_DIR,
        ASR_DIR,
        CAPTIONS_DIR,
        EVIDENCE_DIR
    )

print("✅ Shot evidence updated with ASR + captions")


[MERGE] 16 0613 PROJ COLOR CHEF
[MERGE] 6607010_2158732_DE_de_9000D_Winter Sale_Feed&Masthead_YT_Women_Step2_16x9_20s_D - H&M (720p, h264, youtube)
[MERGE] Adidas _Beyond The Blue_ AI Commercial (Midjourney + @RunwayML + @topazlabs )  AI Advertising
[MERGE] All the Best Moments are Better With Pepsi
[MERGE] Banned M&M's Commercial
[MERGE] BORZ WEAR (Streetwear Commercial Video) - ICE SQUAD MEDIA
[MERGE] CINEMATIC PRODUCT VIDEO _ Go Good Drinks
[MERGE] Clothing commercial video _ Jacferdi _ Fujifilm X-T3
[MERGE] Essentials Clothing Shoot _ Promotional Video
[MERGE] Every Table Has A Story
[MERGE] Gucci Presents_ Gucci Oud, the new unisex fragrance
[MERGE] Introducting ACQUA DI GIÒ PROFONDO PARFUM by Giorgio Armani
[MERGE] Kellogg’s Multigrain Chocos _ Multigrain Energy, More Chocolatey _ Hindi 30 sec
[MERGE] KFC Hot and Cheesy Chicken!
[MERGE] MANGO Committed _ Making FASHION more SUSTAINABLE
[MERGE] McDonald's AI Commercial_ A Taste of Tomorrow
[MERGE] Nike - Running Isn't Just Running